In [ ]:
import torch
import torchaudio
import torchaudio.functional as F
import torch.nn.functional as FT
import matplotlib.pyplot as plt
import numpy as np
import random
import os
from data_converter import DataConverter
from transformers import AutoProcessor, AutoModel
import cv2

wav_2_vec_bundle = torchaudio.pipelines.WAV2VEC2_BASE
wav_2_vec_model = wav_2_vec_bundle.get_model()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
wav_2_vec_model = wav_2_vec_model.to(device)
converter = DataConverter()

def wav_2_vec(file):
    waveform, sr = torchaudio.load(file)
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)
    with torch.no_grad():
        features, _ = wav_2_vec_model(waveform)
    print(features.shape)
    return features, waveform

def create_tssm(file, histogram_equalisation=False):
    features, waveform = wav_2_vec(file)
    x = features[0]
    
    x=FT.normalize(x, p =2, dim =-1)
    tssm = x @ x.T
    tssm = tssm.numpy()

    if histogram_equalisation:
        tssm = converter.apply_histogram_equalisation(tssm)
    return tssm


In [ ]:
#visualise tssm of random wav file

folder = "/scratch/local/ssd/hani/RVN/wav/train/"

files = os.listdir(folder)
random_file = random.choice(files)
print(random_file)
tssm = create_tssm(os.path.join(folder, random_file), histogram_equalisation=True)
plt.imshow(tssm, cmap='magma', interpolation='nearest')
plt.colorbar()
plt.show()


In [ ]:
input_folder="/scratch/local/ssd/hani/RVN-wav/test/"
output_folder="/scratch/local/ssd/hani/RVN-tssm/test/"

choice = random.choice(os.listdir(input_folder))
print(f"Processing file: {choice}")
file = os.path.join(input_folder, choice)

rep_choice = random.choice(os.listdir("/scratch/local/ssd/hani/FSD50K/train/"))
rep_choice = "361396.wav"
print(f"Using repetition file: {rep_choice}")
#print length of rep_choice file
print(f"Length of repetition file: {torchaudio.info(os.path.join('/scratch/local/ssd/hani/FSD50K/train/', rep_choice)).num_frames / torchaudio.info(os.path.join('/scratch/local/ssd/hani/FSD50K/train/', rep_choice)).sample_rate} seconds")
rep_filepath = os.path.join("/scratch/local/ssd/hani/FSD50K/train/", rep_choice)


In [2]:
rep_filepath="buildup.wav"

In [10]:
#CHOOSE TEST FILE ALREADY MADE
file = "audio_strong/TEST.wav"

In [48]:
#CREATE TOY AUDIO

converter = DataConverter(musan_options=["noise"])
y, sr, num_repetitions, _ = converter.create_augmented_wav(rep_filepath, 10.0, converter.max_repetitions, pause_between_reps=True, looped_wav=False, forced_repetitions=3, forced_time_window=1.0)
torchaudio.save("toy_audio.wav", y, sr)
file = "toy_audio.wav"

In [376]:
y_noise = converter.add_musan_noise(y, sr, snr_db_range=(23, 30))
torchaudio.save("toy_audio.wav", y_noise, sr)

In [ ]:
#VISUALISE WAVEFORM AND ZEROES

y_channel = y[0].numpy()
threshold = 1e-5

zeros_idx = np.where(np.abs(y_channel) < threshold)[0]

# plt.figure(figsize=(15,4))
plt.figure(figsize=(15,2))
plt.plot(y_channel, color='Blue')
# plt.scatter(zeros_idx, y_channel[zeros_idx], color='red', marker='x', s=30, label=f'Near-zero (<{threshold})', zorder=5)
plt.xlabel('Sample')
plt.ylabel('Amplitude')
plt.title('Waveform with near-zero points marked')
plt.legend()
plt.show()

In [11]:
# WAV2VEC2
features, _ = wav_2_vec(file)

print(features.shape)  # (batch, time, feature)

torch.Size([1, 499, 768])
torch.Size([1, 499, 768])


In [192]:
#AV HUBERT

import shutil

shutil.copy(file, "file.wav")

'file.wav'

In [ ]:
# file = "/users/hani/dplusn.wav"
# file = "/scratch/local/ssd/hani/RS/wav/test/009200.wav"
# file = "/scratch/local/ssd/hani/RSN/wav/test/011752.wav"
# file = "/scratch/local/ssd/hani/RVN/wav/test/002162.wav"
# file = "/scratch/local/hdd/hani/bbc_clocks/audio/07016229.wav"
# file = "/scratch/local/hdd/hani/heartbeats/wav/b0479.wav"
file = "/scratch/local/hdd/hani/dolphins/test_padded/PulseTrain_755.wav"
file = "/scratch/local/hdd/hani/dolphins/test_padded/PulseTrain_092.wav"
tssm = create_tssm(file, histogram_equalisation=False)
print(np.max(tssm), np.min(tssm))

#apply gaussian blur to tssm
tssm = cv2.GaussianBlur(tssm, (3, 3), 0)
#visualise tssm

plt.imshow(tssm, cmap='magma', interpolation='nearest')
#no axes, just the image
plt.colorbar()
plt.axis('off')
plt.show()

In [ ]:
#batch process and save tssm files

no_of_files = len(os.listdir(input_folder))
step_size = max(1, no_of_files // 500)

for i in range(0, no_of_files):
    item = os.listdir(input_folder)[i]
    if i % step_size == 0:
        print(f"Processing file {i+1} / {no_of_files}: {item}")
    file = os.path.join(input_folder, item)
    
    tssm = create_tssm(file, histogram_equalisation=True)
    save_path = os.path.join(output_folder, item.replace('.wav', '.npy'))
    np.save(save_path, tssm)

In [ ]:
#visualise random tssm file
folder = "/scratch/local/ssd/hani/audioset_eval/tssm/test/"

# folder = "/scratch/local/ssd/hani/extreme-relabelled-tssm/test/"
# folder = "/scratch/local/ssd/hani/countix-av-tssm/test/"

random_tssm_file = random.choice(os.listdir(folder))
tssm = np.load(os.path.join(folder, random_tssm_file))
print(f"Visualizing TSSM file: {random_tssm_file}")
print(f"Class label: {random_tssm_file.split('_')[-1].replace('.npy','')}")
plt.imshow(tssm, cmap='magma', interpolation='nearest')
plt.show()

print(np.max(tssm), np.min(tssm))

In [ ]:
#visualise tssm of specific file
file = "/scratch/local/ssd/hani/RSN/wav/train/000567.wav"
tssm = create_tssm(file, histogram_equalisation=False)
plt.imshow(tssm, cmap='magma', interpolation='nearest')
plt.colorbar()
plt.show()

#now without axes
plt.imshow(tssm, cmap='magma', interpolation='nearest')
plt.axis('off')
plt.show()
